In [ ]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/functions/across_cohort_descriptive_func.R')
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/functions/ind_descriptive_func.R')
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/functions/cci_function.R')


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor




In [2]:
final_pid <- read.delim(
    '/home/jupyter/workspaces/result/final/final_pid.txt',
    as.is = T,
    stringsAsFactors = F
    );
table(final_pid$cohort)
dim(final_pid)
final_pid <- as.data.frame(final_pid)


Diabetes  Healthy Insomnia Prostate 
    1407     6960     1542      256 

[1] 10165     2

In [11]:
library(VennDiagram)

venn.diagram(
  x = list(
      final_pid$person_id[which(final_pid$cohort == 'Diabetes')],
      final_pid$person_id[which(final_pid$cohort == 'Insomnia')],
      final_pid$person_id[which(final_pid$cohort == 'Prostate')],
      final_pid$person_id[which(final_pid$cohort == 'Healthy')]
          ),
  category.names = c('Diabetes', 'Insomnia', 'Prostate', 'Healthy'),
  filename = '/home/jupyter/workspaces/result/final/final_venn.png',
  output=TRUE
)

ERROR: Error in venn.diagram(x = list(final_pid$person_id[which(final_pid$cohort == : argument "filename" is missing, with no default


In [13]:
# read data 
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Diabetes'),'person_id'], collapse = ", ")
diabetes_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", diabetes.pid.sql, ")"
)
diabetes_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_meta",
  "diabetes_meta_*.csv")
message(str_glue('The data will be written to {diabetes_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260506/diabetes_meta/diabetes_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [15]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_meta/diabetes_meta_*.csv'

diabetes_meta <- read_bq_export_from_workspace_bucket(diabetes_meta_path)

dim(diabetes_meta)

head(diabetes_meta, 5) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260506/diabetes_meta/diabetes_meta_000000000000.csv.



[1] 1407   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
9419238,1963-06-15 00:00:00 UTC,38003615,Middle Eastern or North African,38003564,Not Hispanic or Latino,45878463,Female,38003615,Middle Eastern or North African
4537320,1964-06-15 00:00:00 UTC,38003615,Middle Eastern or North African,38003564,Not Hispanic or Latino,45878463,Female,38003615,Middle Eastern or North African
7867772,1995-06-15 00:00:00 UTC,38003615,Middle Eastern or North African,38003564,Not Hispanic or Latino,45880669,Male,38003615,Middle Eastern or North African
9077883,1975-06-15 00:00:00 UTC,38003615,Middle Eastern or North African,38003564,Not Hispanic or Latino,45880669,Male,38003615,Middle Eastern or North African
2356871,1985-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these


In [18]:
demographic_tbl_func(df = diabetes_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "1,407"      
 2 Age                              ""           
 3 18-44                            "157 (11%)"  
 4 45-64                            "552 (39%)"  
 5 >65                              "698 (50%)"  
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "128 (9%)"   
 8 Not Hispanic or Latino           "1,240 (88%)"
 9 Others                           "39 (3%)"    
10 Race                             ""           
11 American Indian or Alaska Native "9 (1%)"     
12 Asian                            "44 (3%)"    
13 Black or African American        "147 (10%)"  
14 Middle Eastern or North African  "5 (<1%)"    
15 More than one population         "99 (7%)"    
16 Others                           "115 (8%)"   
17 White                            "988 (70%)"  
18 Sex at birth                

NULL

In [19]:
table(diabetes_meta$ethnicity_concept_id)


  903079   903096  1586148 38003563 38003564 
       1       19       19      128     1240 

In [20]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Insomnia'),'person_id'], collapse = ", ")

insomnia_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", insomnia.pid.sql, ")"
)
insomnia_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_meta",
  "insomnia_meta_*.csv")
message(str_glue('The data will be written to {insomnia_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260506/insomnia_meta/insomnia_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [21]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_meta/insomnia_meta_*.csv'

insomnia_meta <- read_bq_export_from_workspace_bucket(insomnia_meta_path)

dim(insomnia_meta)

head(insomnia_meta, 5) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260506/insomnia_meta/insomnia_meta_000000000000.csv.



[1] 1542   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1617237,1964-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
1829801,1983-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
7659537,1949-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
2081130,1971-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
3471584,1958-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population


In [22]:
demographic_tbl_func(df = insomnia_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "1,542"      
 2 Age                              ""           
 3 18-44                            "275 (18%)"  
 4 45-64                            "521 (34%)"  
 5 >65                              "746 (48%)"  
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "115 (7%)"   
 8 Not Hispanic or Latino           "1,401 (91%)"
 9 Others                           "26 (2%)"    
10 Race                             ""           
11 American Indian or Alaska Native "11 (1%)"    
12 Asian                            "27 (2%)"    
13 Black or African American        "90 (6%)"    
14 Middle Eastern or North African  "8 (1%)"     
15 More than one population         "85 (6%)"    
16 Others                           "98 (6%)"    
17 White                            "1,223 (79%)"
18 Sex at birth                

NULL

In [23]:
# prostate read data 
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Prostate'),'person_id'], collapse = ", ")

pca_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", pca.pid.sql, ")"
)
pca_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_meta",
  "pca_meta_*.csv")
message(str_glue('The data will be written to {pca_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260506/pca_meta/pca_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [25]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_meta/pca_meta_*.csv'

pca_meta <- read_bq_export_from_workspace_bucket(pca_meta_path)

dim(pca_meta)

head(pca_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260506/pca_meta/pca_meta_000000000000.csv.



[1] 256  10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1088070,1946-06-15 00:00:00 UTC,8516,Black or African American,38003564,Not Hispanic or Latino,45880669,Male,8516,Black or African American
1967823,1956-06-15 00:00:00 UTC,8516,Black or African American,38003564,Not Hispanic or Latino,45880669,Male,8516,Black or African American
8702495,1951-06-15 00:00:00 UTC,8516,Black or African American,38003564,Not Hispanic or Latino,45880669,Male,8516,Black or African American
2414321,1954-06-15 00:00:00 UTC,8516,Black or African American,38003564,Not Hispanic or Latino,45880669,Male,8516,Black or African American
9013612,1951-06-15 00:00:00 UTC,8516,Black or African American,38003564,Not Hispanic or Latino,45880669,Male,8516,Black or African American


In [26]:
demographic_tbl_func(df = pca_meta)

# A tibble: 18 × 2
   Demographic                      Value       
   <chr>                            <chr>       
 1 N                                "256"       
 2 Age                              ""          
 3 45-64                            "11 (4%)"   
 4 >65                              "245 (96%)" 
 5 Ethnicity                        ""          
 6 Hispanic or Latino               "10 (4%)"   
 7 Not Hispanic or Latino           "242 (95%)" 
 8 Others                           "4 (2%)"    
 9 Race                             ""          
10 American Indian or Alaska Native "1 (<1%)"   
11 Asian                            "3 (1%)"    
12 Black or African American        "11 (4%)"   
13 Middle Eastern or North African  "1 (<1%)"   
14 More than one population         "5 (2%)"    
15 Others                           "11 (4%)"   
16 White                            "224 (88%)" 
17 Sex at birth                     ""          
18 Male                             "256 (100%)"


NULL

In [27]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Healthy'),'person_id'], collapse = ", ")

healthy_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", healthy.pid.sql, ")"
)
healthy_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_meta",
  "healthy_meta_*.csv")
message(str_glue('The data will be written to {healthy_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260506/healthy_meta/healthy_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [28]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_meta/healthy_meta_*.csv'

healthy_meta <- read_bq_export_from_workspace_bucket(healthy_meta_path)

dim(healthy_meta)

head(healthy_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260506/healthy_meta/healthy_meta_000000000000.csv.



[1] 6960   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1188079,1981-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
1346705,1998-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
3043509,1997-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
2523178,1992-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
6105989,1978-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these


In [29]:
demographic_tbl_func(df = healthy_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "6,960"      
 2 Age                              ""           
 3 18-44                            "1,928 (28%)"
 4 45-64                            "2,294 (33%)"
 5 >65                              "2,738 (39%)"
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "840 (12%)"  
 8 Not Hispanic or Latino           "5,959 (86%)"
 9 Others                           "161 (2%)"   
10 Race                             ""           
11 American Indian or Alaska Native "49 (1%)"    
12 Asian                            "417 (6%)"   
13 Black or African American        "366 (5%)"   
14 Middle Eastern or North African  "34 (<1%)"   
15 More than one population         "513 (7%)"   
16 Others                           "680 (10%)"  
17 White                            "4,901 (70%)"
18 Sex at birth                

NULL

In [30]:
diabetes_meta$cohort <- 'Diabetes'
insomnia_meta$cohort <- 'Insomnia'
pca_meta$cohort <- 'Prostate'
healthy_meta$cohort <- 'Healthy'

meta_all <- rbind(diabetes_meta,insomnia_meta,pca_meta,healthy_meta)
meta_all$age <- as.integer(floor(interval(meta_all$date_of_birth, Sys.Date()) / years(1)))
meta_all[1:3,]

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category,cohort,age
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<int>
9419238,1963-06-15 00:00:00 UTC,38003615,Middle Eastern or North African,38003564,Not Hispanic or Latino,45878463,Female,38003615,Middle Eastern or North African,Diabetes,62
4537320,1964-06-15 00:00:00 UTC,38003615,Middle Eastern or North African,38003564,Not Hispanic or Latino,45878463,Female,38003615,Middle Eastern or North African,Diabetes,61
7867772,1995-06-15 00:00:00 UTC,38003615,Middle Eastern or North African,38003564,Not Hispanic or Latino,45880669,Male,38003615,Middle Eastern or North African,Diabetes,30


In [50]:
# CCI
cci <- read.delim(
    '/home/jupyter/workspaces/result/pre_final/pre_final_cci.txt',
    as.is = T,
    stringsAsFactors = F
    );
cci_final <- cci[which(cci$person_id %in% final_pid$person_id),]
dim(cci_final)
length(final_pid$person_id)

[1] 10165     3

[1] 10165

   person_id            age          age_points   
 Min.   :1000464   Min.   :21.00   Min.   :0.000  
 1st Qu.:2285677   1st Qu.:43.00   1st Qu.:0.000  
 Median :4257414   Median :59.00   Median :1.000  
 Mean   :4753837   Mean   :56.81   Mean   :1.478  
 3rd Qu.:7090724   3rd Qu.:71.00   3rd Qu.:3.000  
 Max.   :9997301   Max.   :98.00   Max.   :4.000  

    cohort            person_id     cci_score  
 Length:0           Min.   : NA   Min.   : NA  
 Class :character   1st Qu.: NA   1st Qu.: NA  
 Mode  :character   Median : NA   Median : NA  
                    Mean   :NaN   Mean   :NaN  
                    3rd Qu.: NA   3rd Qu.: NA  
                    Max.   : NA   Max.   : NA  

In [51]:
final_demographic_plot_age_func(df = meta_all, path.out = '/home/jupyter/workspaces/result/final/AcrossCohort/')

NULL

In [54]:

final_demographic_plot_cii_func(cci_final, path.out = '/home/jupyter/workspaces/result/final/AcrossCohort/')

# A tibble: 14 × 6
   cohort   cci_category   n_cohort median_cci n_category   pct
   <chr>    <fct>             <int>      <dbl>      <int> <dbl>
 1 Diabetes None (0)           1407          5         19   1.4
 2 Diabetes Mild (1-2)         1407          5        190  13.5
 3 Diabetes Moderate (3-4)     1407          5        379  26.9
 4 Diabetes Severe (≥5)        1407          5        819  58.2
 5 Healthy  None (0)           6960          1       2480  35.6
 6 Healthy  Mild (1-2)         6960          1       2491  35.8
 7 Healthy  Moderate (3-4)     6960          1       1989  28.6
 8 Insomnia None (0)           1542          3        161  10.4
 9 Insomnia Mild (1-2)         1542          3        374  24.3
10 Insomnia Moderate (3-4)     1542          3        481  31.2
11 Insomnia Severe (≥5)        1542          3        526  34.1
12 Prostate Mild (1-2)          256          4         16   6.2
13 Prostate Moderate (3-4)      256          4        123  48  
14 Prostate Severe (≥

NULL

    cohort            person_id         cci_score     
 Length:10413       Min.   :1000464   Min.   : 0.000  
 Class :character   1st Qu.:2107936   1st Qu.: 0.000  
 Mode  :character   Median :3828013   Median : 2.000  
                    Mean   :4573109   Mean   : 2.661  
                    3rd Qu.:6911215   3rd Qu.: 4.000  
                    Max.   :9997301   Max.   :20.000  